# Event Picker

This notebook allows to inspect the results from the clustering and estimation phases of Spyral, by displaying the locations of the tracks on the kinematics, PID, and excitation-energy vs vertex-z plots, and showing the point cloud with the cluster determination. The clustering phase groups points from the point cloud into clusters, and the estimation phase determines the kinematical and energy loss for each of these clusters. If the estimation phase fails for a particular cluster, there is no corresponding track data recorded and the cluster is simply dropped. This explains why there can be more clusters than tracks. Tracks for which the InterpSolver phase failed (no fitted vertex / brho) are still shown in the kinematics and PID plots, but cannot appear in the excitation-energy vs vertex-z plot — when one of them is selected the panel reports `solver: failed`.

When clicking somewhere on the kinematics, PID, or ex-vs-vz plot, the closest data point of a particular track is picked and the event to which it belongs is displayed, as well as the data from other tracks of that event if there are any. The color scheme is consistent between all the plots, which helps in identifying which particles are involved in a given event.

This version supports loading **multiple runs at once** by specifying a `run_min`/`run_max` range. Because event numbers restart at 1 in every file, each track is identified by the `(run, event)` pair internally, so picking always lands on the correct event in the correct run. The workspace path where all the Spyral files are located also needs to be specified, and the physics configuration (nuclei, target material, beam energy) must match the one used in `example_analysis.ipynb` for the excitation-energy calculation to make sense.

In [ ]:
from spyral.core.point_cloud import PointCloud
from spyral.core.cluster import Cluster

import h5py as h5
import polars as pl
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

%matplotlib widget
plt.close()

workspace_path = Path("/home/junrui/Ptest/Spyral_workspace/v1.0/workspacedd")
cloud_path = workspace_path / "Pointcloud"
cluster_path = workspace_path / "Cluster"
estimation_result_path = workspace_path / "Estimation"
solver_result_path = workspace_path / "InterpSolver"  # used later for ex / vertex_z

# Range of runs to load (inclusive). Event numbers restart at 1 in each file,
# so internally tracks are identified by the (run, event) pair.
run_min = 7
run_max = 7

# Open the per-run pointcloud and cluster h5 files and load the per-run
# Estimation parquets. Files are kept open in dicts keyed by run number so
# the picker can fetch the right cloud / cluster on demand.
cloud_files: dict[int, h5.File] = {}
cluster_files: dict[int, h5.File] = {}
cloud_groups: dict[int, h5.Group] = {}
cluster_groups: dict[int, h5.Group] = {}

estimation_frames: list[pl.DataFrame] = []
loaded_runs: list[int] = []
for run in range(run_min, run_max + 1):
    est_path = estimation_result_path / f"run_{run:04d}.parquet"
    cloud_file_path = cloud_path / f"run_{run:04d}.h5"
    cluster_file_path = cluster_path / f"run_{run:04d}.h5"
    if not (est_path.exists() and cloud_file_path.exists() and cluster_file_path.exists()):
        print(f"Skipping run {run}: missing one of estimation/cloud/cluster files")
        continue

    cloud_files[run] = h5.File(cloud_file_path, "r")
    cluster_files[run] = h5.File(cluster_file_path, "r")
    cloud_groups[run] = cloud_files[run].get("cloud")
    cluster_groups[run] = cluster_files[run].get("cluster")

    run_df = pl.scan_parquet(est_path).collect()
    run_df = run_df.with_columns(pl.lit(run, dtype=pl.Int64).alias("run"))
    estimation_frames.append(run_df)
    loaded_runs.append(run)

if not estimation_frames:
    raise RuntimeError(f"No runs loaded in range [{run_min}, {run_max}]")

df = pl.concat(estimation_frames, how="diagonal_relaxed")
print(f"Loaded {len(loaded_runs)} runs: {loaded_runs}")
print(f"Total tracks (Estimation rows): {df.height}")


## Physics configuration

The settings below mirror those in `example_analysis.ipynb` and are needed to compute the residual excitation energy used by the `ex` vs `vertex_z` plot. Keep them in sync with the analysis notebook for the same dataset.

In [ ]:
from spyral.core.constants import AMU_2_MEV, QBRHO_2_P
from spyral_utils.nuclear import NuclearDataMap
from spyral_utils.nuclear.target import GasTarget, load_target, SolidTarget
import vector

nuclear_map = NuclearDataMap()

# IC entrance/exit window, AT-TPC entrance window (ug/cm^2)
ic_window_material = SolidTarget(compound=[[1,1,14],[6,12,14],[7,14,4],[8,16,4]], thickness=1422.312, nuclear_data=nuclear_map)
attpc_window_material = SolidTarget(compound=[[1,1,14],[6,12,14],[7,14,4],[8,16,4]], thickness=1422.312, nuclear_data=nuclear_map)
# IC gas (Torr)
ic_gas_material = GasTarget(compound=[(6,12,1),(9,18,4)], pressure=200.0, nuclear_data=nuclear_map)
ic_gas_thickness = 0.035  # m

# AT-TPC target gas material
target_material_path = Path("/home/junrui/Ptest/Spyral-1.0/PID/compound_dd.json")

# Reaction nuclei (must match example_analysis.ipynb)
ejectile_z, ejectile_a = 1, 1     # detected
projectile_z, projectile_a = 6, 16  # beam
target_z, target_a = 1, 2           # target

residual_z = target_z + projectile_z - ejectile_z
residual_a = target_a + projectile_a - ejectile_a

target_material = load_target(target_material_path, nuclear_map)
if not isinstance(target_material, GasTarget):
    raise RuntimeError("Target material must be a GasTarget")

ejectile = nuclear_map.get_data(ejectile_z, ejectile_a)
projectile = nuclear_map.get_data(projectile_z, projectile_a)
target = nuclear_map.get_data(target_z, target_a)
residual = nuclear_map.get_data(residual_z, residual_a)
print(f"Reaction: {target}({projectile}, {ejectile}){residual}")

# Beam energy bookkeeping
proj_energy_accel = 196.0  # MeV
proj_energy_ic = proj_energy_accel - ic_window_material.get_energy_loss(projectile, proj_energy_accel, np.array([0.0]))[0]
proj_energy_ic_exit = proj_energy_ic - ic_gas_material.get_energy_loss(projectile, proj_energy_ic, np.array([ic_gas_thickness]))[0]
proj_energy_post_ic = proj_energy_ic_exit - ic_window_material.get_energy_loss(projectile, proj_energy_ic_exit, np.array([0.0]))[0]
proj_energy_start = proj_energy_post_ic - attpc_window_material.get_energy_loss(projectile, proj_energy_post_ic, np.array([0.0]))[0]
print(f"Beam energy entering AT-TPC gas: {proj_energy_start:.3f} MeV")


## Load InterpSolver fits and compute excitation energy

For each loaded run we read the corresponding `InterpSolver/run_XXXX_<ejectile>.parquet`, compute the residual excitation energy from the fitted vertex / brho / polar (same formula as `example_analysis.ipynb`), and left-join the result back onto the Estimation dataframe on `(run, event, cluster_label)`. Tracks for which the InterpSolver phase had no row receive `ex = NaN` and `vertex_z = NaN`, and are flagged in `solver_ok`.

In [ ]:
target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})

def compute_ex_for_solver_df(solver_df: pl.DataFrame) -> np.ndarray:
    """Excitation energy of the residual for every row of an InterpSolver parquet."""
    vertices = solver_df.select(["vertex_x", "vertex_y", "vertex_z"]).to_numpy()
    distances = np.linalg.norm(vertices, axis=1)
    projectile_ke = proj_energy_start - target_material.get_energy_loss(projectile, proj_energy_start, distances)
    projectile_vec = vector.array({
        "px": np.zeros(len(projectile_ke)),
        "py": np.zeros(len(projectile_ke)),
        "pz": np.sqrt(projectile_ke * (projectile_ke + 2.0 * projectile.mass)),
        "E":  projectile_ke + projectile.mass,
    })
    brho_arr  = solver_df["brho"].to_numpy()
    polar_arr = solver_df["polar"].to_numpy()
    az_arr    = solver_df["azimuthal"].to_numpy()
    momentum  = brho_arr * float(ejectile.Z) * QBRHO_2_P
    ejectile_vec = vector.array({
        "px": momentum * np.sin(polar_arr) * np.cos(az_arr),
        "py": momentum * np.sin(polar_arr) * np.sin(az_arr),
        "pz": momentum * np.cos(polar_arr),
        "E":  np.sqrt(momentum**2.0 + ejectile.mass**2.0),
    })
    residual_vec = target_vector + projectile_vec - ejectile_vec  # type: ignore
    return residual_vec.mass - residual.mass


solver_frames: list[pl.DataFrame] = []
for run in loaded_runs:
    solver_path = solver_result_path / f"run_{run:04d}_{ejectile.isotopic_symbol}.parquet"
    if not solver_path.exists():
        print(f"No InterpSolver parquet for run {run} (looked for {solver_path.name})")
        continue
    sdf = pl.scan_parquet(solver_path).collect()
    if sdf.height == 0:
        continue
    ex_vals = compute_ex_for_solver_df(sdf)
    sdf = sdf.with_columns([
        pl.lit(run, dtype=pl.Int64).alias("run"),
        pl.Series("ex", ex_vals, dtype=pl.Float64),
    ])
    # Keep only the columns we need for the join + plot
    keep_cols = ["run", "event", "cluster_label", "vertex_x", "vertex_y", "vertex_z", "ex", "redchisq"]
    sdf = sdf.select([c for c in keep_cols if c in sdf.columns])
    solver_frames.append(sdf)

if solver_frames:
    solver_df_all = pl.concat(solver_frames, how="diagonal_relaxed")
    df = df.join(solver_df_all, on=["run", "event", "cluster_label"], how="left")
else:
    print("WARNING: no InterpSolver parquets found — ex/vertex_z columns will be all NaN")
    df = df.with_columns([
        pl.lit(None, dtype=pl.Float64).alias("vertex_x"),
        pl.lit(None, dtype=pl.Float64).alias("vertex_y"),
        pl.lit(None, dtype=pl.Float64).alias("vertex_z"),
        pl.lit(None, dtype=pl.Float64).alias("ex"),
        pl.lit(None, dtype=pl.Float64).alias("redchisq"),
    ])

df = df.with_columns(pl.col("ex").is_not_null().alias("solver_ok"))
n_ok = int(df["solver_ok"].sum())
print(f"Tracks with successful InterpSolver fit: {n_ok} / {df.height}")


## Analysis cuts

Define which fitted tracks count as "good". Tracks with `solver_ok=True` that
fail these cuts are still shown in all plots, but in a faded grey style so the
"good" tracks dominate visually. Edit this single dict and re-run the cell —
the plot below picks up the new mask automatically.

In [ ]:
# Edit these to change what counts as a "good" fit. Set any value to None to
# disable that particular cut.
analysis_cuts = dict(
    vertex_z_min  = 0.005,    # m
    vertex_z_max  = 0.995,    # m
    polar_max_deg = 88.0,     # drop tracks with polar > this
    redchisq_max  = None,     # e.g. 5.0 to enable a chi^2 cut
)


def _build_analysis_pass(df: pl.DataFrame, cuts: dict) -> pl.Series:
    """Boolean column: True iff the row has a solver fit AND survives `cuts`."""
    expr = pl.col("ex").is_not_null()  # must have a solver row to begin with
    if cuts.get("vertex_z_min") is not None:
        expr = expr & (pl.col("vertex_z") > cuts["vertex_z_min"])
    if cuts.get("vertex_z_max") is not None:
        expr = expr & (pl.col("vertex_z") < cuts["vertex_z_max"])
    if cuts.get("polar_max_deg") is not None:
        expr = expr & (pl.col("polar") < np.deg2rad(cuts["polar_max_deg"]))
    if cuts.get("redchisq_max") is not None and "redchisq" in df.columns:
        expr = expr & (pl.col("redchisq") < cuts["redchisq_max"])
    return df.select(expr.alias("analysis_pass")).to_series()


df = df.with_columns(_build_analysis_pass(df, analysis_cuts).alias("analysis_pass"))

n_total  = df.height
n_solver = int(df["solver_ok"].sum())
n_pass   = int(df["analysis_pass"].sum())
print(f"Total tracks            : {n_total}")
print(f"Solver succeeded        : {n_solver}")
print(f"Solver + passes cuts    : {n_pass}")
print(f"Solver + fails cuts     : {n_solver - n_pass}")
print(f"Solver failed           : {n_total - n_solver}")


In [ ]:
plt.close()

# ---------------- per-track arrays (concatenated across all runs) ----------------
runs       = df["run"].to_numpy()
ev         = df["event"].to_numpy()
ys         = df["brho"].to_numpy()
xs         = df["polar"].to_numpy()
de         = df["sqrt_dEdx"].to_numpy()
cl         = df["cluster_label"].to_numpy()
ex_arr     = df["ex"].to_numpy()         # NaN where solver failed
vz_arr     = df["vertex_z"].to_numpy()   # NaN where solver failed
solver_ok  = df["solver_ok"].to_numpy().astype(bool)
analysis_pass = df["analysis_pass"].to_numpy().astype(bool)

# Three disjoint tiers (each track belongs to exactly one)
tier_pass   = analysis_pass                    # solver ok AND passes cuts
tier_cut    = solver_ok & ~analysis_pass       # solver ok BUT fails cuts
tier_failed = ~solver_ok                       # no solver row

cmap = plt.cm.Set1

# ---------------- figure layout ----------------
# Top row:    C = kinematics, D = PID, E = ex vs vertex_z
# Bottom row: A = 3D pointcloud, B = 2D pointcloud
fig, axs = plt.subplot_mosaic(
    """
    CDE
    ABB
    """,
    per_subplot_kw={
        "A": {
            "projection": "3d",
            "box_aspect": (1, 1, 1),
            "aspect": "equalxy",
        }
    },
    figsize=(15.0, 11.0),
    constrained_layout=True,
)


# Each pickable artist is registered with the *global* track indices it owns
# and the (x, y) arrays used for nearest-point selection. on_pick simply looks
# the clicked artist up in this table — no per-artist branching needed.
pickable_registry: dict = {}

def add_pickable(ax, mask, x_data, y_data, **plot_kwargs):
    if not mask.any():
        return None
    sub_idx = np.where(mask)[0]
    artist, = ax.plot(x_data[sub_idx], y_data[sub_idx], picker=True, pickradius=5, **plot_kwargs)
    pickable_registry[artist] = (sub_idx, x_data, y_data)
    return artist


# Kinematics panel C
add_pickable(axs["C"], tier_failed, xs, ys,
             marker='x', linestyle='none', color='red',  markersize=2, alpha=0.4, label="solver failed")
add_pickable(axs["C"], tier_cut,    xs, ys,
             marker='.', linestyle='none', color='grey', markersize=2, alpha=0.4, label="cuts failed")
add_pickable(axs["C"], tier_pass,   xs, ys,
             marker=',', linestyle='none', color='black', markeredgecolor='none')
axs["C"].set_title('Kinematics plot')
axs["C"].set_xlabel(r"$\theta$ (rad)")
axs["C"].set_ylabel(r"$B\rho$ (T·m)")
axs["C"].set_xlim(0.0, np.pi)
axs["C"].set_ylim(0.0, 3.0)

# PID panel D
add_pickable(axs["D"], tier_failed, de, ys,
             marker='x', linestyle='none', color='red',  markersize=2, alpha=0.4, label="solver failed")
add_pickable(axs["D"], tier_cut,    de, ys,
             marker='.', linestyle='none', color='grey', markersize=2, alpha=0.4, label="cuts failed")
add_pickable(axs["D"], tier_pass,   de, ys,
             marker=',', linestyle='none', color='black', markeredgecolor='none')
axs["D"].set_title('PID plot')
axs["D"].set_xlabel(r"$\sqrt{dE/dx}$")
axs["D"].set_ylabel(r"$B\rho$ (T·m)")
axs["D"].set_xlim(0.0, 120.0)
axs["D"].set_ylim(0.0, 3.0)

# Excitation energy vs vertex Z, panel E (failed-solver tracks have no ex/vz so are absent)
add_pickable(axs["E"], tier_cut,    ex_arr, vz_arr,
             marker='.', linestyle='none', color='grey', markersize=2, alpha=0.4, label="cuts failed")
add_pickable(axs["E"], tier_pass,   ex_arr, vz_arr,
             marker=',', linestyle='none', color='black', markeredgecolor='none')
axs["E"].set_title('Excitation energy vs vertex Z')
axs["E"].set_xlabel(f"{residual.get_latex_rep()} Ex (MeV)")
axs["E"].set_ylabel("Vertex Z (m)")
axs["E"].set_xlim(-2.0, 5.0)
axs["E"].set_ylim(0.0, 1.0)


class PointBrowser:
    """Click on any of the kinematics / PID / ex-vz plots to display the
    corresponding event's pointcloud and clusters."""

    MAX_TRACKS = 10  # max number of tracks of the selected event we will highlight

    def __init__(self):
        self.lastind = 0

        self.text          = axs["C"].text(0.05, 0.95, 'selected event: none',  transform=axs["C"].transAxes, va='top')
        self.tracks_text   = axs["C"].text(0.05, 0.85, 'number of tracks: 0',   transform=axs["C"].transAxes, va='top')
        self.clusters_text = axs["C"].text(0.05, 0.90, 'number of clusters: 0', transform=axs["C"].transAxes, va='top')
        self.solver_text   = axs["C"].text(0.05, 0.80, '',                      transform=axs["C"].transAxes, va='top')
        self.cuts_text     = axs["C"].text(0.05, 0.75, '',                      transform=axs["C"].transAxes, va='top')

        # Per-track highlight markers, one set per pickable plot.
        self.marker_C: list = []
        self.marker_D: list = []
        self.marker_E: list = []
        for i in range(self.MAX_TRACKS):
            color = cmap(float(i) / 9.0)
            mC, = axs["C"].plot([xs[0]], [ys[0]], 'o', ms=5, alpha=0.7, color=color, visible=False)
            mD, = axs["D"].plot([de[0]], [ys[0]], 'o', ms=5, alpha=0.7, color=color, visible=False)
            mE, = axs["E"].plot([0.0], [0.0],     'o', ms=5, alpha=0.7, color=color, visible=False)
            self.marker_C.append(mC)
            self.marker_D.append(mD)
            self.marker_E.append(mE)

    def on_pick(self, event):
        info = pickable_registry.get(event.artist)
        if info is None:
            return True
        sub_idx, x_data, y_data = info
        if len(event.ind) == 0:
            return True
        # Pick the data point closest to the actual mouse click within those
        # matplotlib reported as candidates.
        local = event.ind
        chosen = sub_idx[local]
        x = event.mouseevent.xdata
        y = event.mouseevent.ydata
        distances = np.hypot(x - x_data[chosen], y - y_data[chosen])
        self.lastind = int(chosen[distances.argmin()])
        self.update()

    def update(self):
        if self.lastind is None:
            return
        run = int(runs[self.lastind])
        event = int(ev[self.lastind])
        self.update_plots(run, event)

    def update_plots(self, run: int, event: int):
        cloud_group = cloud_groups[run]
        cluster_group = cluster_groups[run]

        event_data = cloud_group[f'cloud_{event}']
        cloud = PointCloud(event, event_data[:].copy())

        axs["A"].clear()
        axs["B"].clear()
        axs["A"].scatter(cloud.data[:, 2], cloud.data[:, 0], cloud.data[:, 1], c=cloud.data[:, 3], s=1, label="Pointcloud")
        axs["A"].set_xlim3d(0., 1000.0)
        axs["A"].set_ylim3d(-300.0, 300.0)
        axs["A"].set_zlim3d(-300.0, 300.0)
        axs["A"].set_title(f"run {run} / event {event}")
        axs["B"].scatter(cloud.data[:, 0], cloud.data[:, 1], c=cloud.data[:, 3], s=1, label="Pointcloud")
        axs["B"].set_xlim(-300.0, 300.0)
        axs["B"].set_ylim(-300.0, 300.0)

        self.text.set_text(f'selected event: run {run} / event {event}')
        same_event = (runs == run) & (ev == event)
        ind = np.where(same_event)[0]
        ntracks = len(ind)
        self.tracks_text.set_text(f'number of tracks: {ntracks}')

        # Status of the picked track
        if solver_ok[self.lastind]:
            self.solver_text.set_text(
                f'solver: ok (ex={ex_arr[self.lastind]:.3f} MeV, vz={vz_arr[self.lastind]:.3f} m)')
        else:
            self.solver_text.set_text('solver: failed')
        if analysis_pass[self.lastind]:
            self.cuts_text.set_text('cuts: passed')
        elif solver_ok[self.lastind]:
            self.cuts_text.set_text('cuts: failed')
        else:
            self.cuts_text.set_text('cuts: n/a')

        for i in range(min(ntracks, self.MAX_TRACKS)):
            color = cmap(float(cl[ind[i]]) / 9.0)
            self.marker_C[i].set_data([xs[ind[i]]], [ys[ind[i]]])
            self.marker_C[i].set_color(color)
            self.marker_C[i].set_visible(True)
            self.marker_C[i].set_label(f"Track {i}")

            self.marker_D[i].set_data([de[ind[i]]], [ys[ind[i]]])
            self.marker_D[i].set_color(color)
            self.marker_D[i].set_visible(True)
            self.marker_D[i].set_label(f"Track {i}")

            if solver_ok[ind[i]]:
                self.marker_E[i].set_data([ex_arr[ind[i]]], [vz_arr[ind[i]]])
                self.marker_E[i].set_color(color)
                self.marker_E[i].set_visible(True)
                self.marker_E[i].set_label(f"Track {i}")
            else:
                self.marker_E[i].set_visible(False)
                self.marker_E[i].set_label(f"_Track {i}")
        for i in range(ntracks, self.MAX_TRACKS):
            self.marker_C[i].set_visible(False)
            self.marker_D[i].set_visible(False)
            self.marker_E[i].set_visible(False)
            self.marker_C[i].set_label(f"_Track {i}")
            self.marker_D[i].set_label(f"_Track {i}")
            self.marker_E[i].set_label(f"_Track {i}")

        axs["C"].legend()
        axs["D"].legend()
        axs["E"].legend()

        clusters = cluster_group[f'event_{event}']
        nclusters = clusters.attrs["nclusters"]
        self.clusters_text.set_text(f'number of clusters: {nclusters}')
        for i in range(nclusters):
            cluster = clusters[f'cluster_{i}']
            cluster_cloud = cluster['cloud']
            cluster_data = Cluster(event, i, cluster_cloud[:].copy())
            color = cmap(float(i) / 9.0)
            axs["A"].scatter(cluster_data.data[:, 2], cluster_data.data[:, 0], cluster_data.data[:, 1],
                             color=color, s=3, label=f"Cluster {cluster_data.label}")
            axs["B"].scatter(cluster_data.data[:, 0], cluster_data.data[:, 1],
                             color=color, s=3, label=f"Cluster {cluster_data.label}")
        axs["A"].legend()
        axs["B"].legend()

        fig.canvas.draw()


browser = PointBrowser()
fig.canvas.mpl_connect('pick_event', browser.on_pick)

# Use the first loaded run / first event in that run for the initial display
first_run = loaded_runs[0]
first_event = int(ev[runs == first_run][0])
browser.update_plots(first_run, first_event)

plt.show()
